# ELENDRIL V8-X1 — Relational Evidence Reader & D2 Paired Screen (Kaggle 2xT4)

Execution notebook pinned to immutable runtime commit:
- `SOURCE_SHA = "6c684820c0c71a5f8bc019dbff3e915381b8b13c"`
- Sessions: `A0` (P vs R), `A1` (F vs F-cap), `A2` (R-noedge vs P-capR), `A3` (B0 vs J), `D0` (PAIR-CE), `D1` (PAIR-EQ), `D2` (PAIR-EQ-CF)
- Change **only `OWNER_AUTHORIZED = True`** when launching an authorized 2xT4 cloud run.


In [ ]:
# Owner-controlled switches (keep OWNER_AUTHORIZED = False during dry-run/CI; set True to run 2xT4)
OWNER_AUTHORIZED = False
SESSION = "A0"  # A0 | A1 | A2 | A3 | D0 | D1 | D2 | U0 | U1
MINUTES = 330
DEVICES = "0,1"
RESUME = False
OPTIONAL_INPUT_DIR = ""
SOURCE_SHA = "6c684820c0c71a5f8bc019dbff3e915381b8b13c"
REPO_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"


In [ ]:
import os, shutil, subprocess, sys, tarfile
from pathlib import Path

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"

WORK_ROOT = Path("/tmp/elendril_x1")
REPO_DIR = WORK_ROOT / "repo"
RUN_DIR = WORK_ROOT / f"session_{SESSION}"
OUTPUT_DIR = Path("/kaggle/working")

if not OWNER_AUTHORIZED:
    print("NOT LAUNCHED: set OWNER_AUTHORIZED = True in the configuration cell to execute X1.")
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    local_tar = Path(f"/kaggle/input/elendril-x1-source/{SOURCE_SHA}.tar.gz")
    if local_tar.exists():
        tar_path = WORK_ROOT / f"{SOURCE_SHA}.tar.gz"
        shutil.copy2(local_tar, tar_path)
        with tarfile.open(tar_path, "r:gz") as tf:
            tf.extractall(REPO_DIR)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "init", "-q"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "remote", "add", "origin", REPO_URL], check=True)
        try:
            subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth=1", "origin", SOURCE_SHA], check=True)
        except subprocess.CalledProcessError:
            subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", SOURCE_SHA], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", SOURCE_SHA], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    prev_pp = os.environ.get("PYTHONPATH", "")
    pp_parts = [p for p in prev_pp.split(os.pathsep) if p] if prev_pp else []
    if str(REPO_DIR) not in pp_parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_DIR), *pp_parts]) if pp_parts else str(REPO_DIR)
    print("Checked out immutable X1 runtime:", SOURCE_SHA)


In [ ]:
import torch
if not OWNER_AUTHORIZED:
    print("NOT LAUNCHED: set OWNER_AUTHORIZED = True")
else:
    gpu_count = torch.cuda.device_count()
    print("Torch:", torch.__version__, "CUDA GPUs:", gpu_count)
    for idx in range(gpu_count):
        print(f"  cuda:{idx} -> {torch.cuda.get_device_name(idx)}")
    assert torch.cuda.is_available() and gpu_count >= 2, f"2xT4 CUDA GPUs required, found {gpu_count}"


In [ ]:
import os, subprocess, sys
if not OWNER_AUTHORIZED:
    print("NOT LAUNCHED: set OWNER_AUTHORIZED = True")
else:
    env = dict(os.environ)
    env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    env["PYTHONDONTWRITEBYTECODE"] = "1"
    env["PYTHONUNBUFFERED"] = "1"
    env["OMP_NUM_THREADS"] = "2"
    env["MKL_NUM_THREADS"] = "2"
    common = ["--source", SOURCE_SHA, "--session", SESSION, "--work-dir", str(RUN_DIR)]
    for sub_args in (
        ["prepare", *common],
        ["validate", *common, "--cpu-fixtures"],
        ["admit", *common, "--devices", DEVICES, "--max-minutes", str(MINUTES)],
    ):
        subprocess.run([sys.executable, "-m", "tools.elendril_v8x_campaign", *sub_args], env=env, cwd=str(REPO_DIR), check=True)
    run_cmd = [
        sys.executable, "-m", "tools.elendril_v8x_campaign",
        "resume" if RESUME else "run",
        *common,
        "--devices", DEVICES,
        "--max-minutes", str(MINUTES),
        "--output-dir", str(OUTPUT_DIR),
        "--owner-authorized",
    ]
    subprocess.run(run_cmd, env=env, cwd=str(REPO_DIR), check=True)


In [ ]:
import json, sys
if not OWNER_AUTHORIZED:
    print("NOT LAUNCHED: set OWNER_AUTHORIZED = True")
else:
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    from IPython.display import FileLink, display
    from tools.elendril_v8x_export import analysis_zip_name, full_zip_name, verify_x1_export

    receipt = verify_x1_export(OUTPUT_DIR, session=SESSION, expected_source_commit=SOURCE_SHA)
    print("Verified X1 exports:", json.dumps(receipt, indent=2))
    for fname in ("FINAL_STATUS.json", analysis_zip_name(SESSION), full_zip_name(SESSION)):
        if (OUTPUT_DIR / fname).exists():
            display(FileLink(str(OUTPUT_DIR / fname)))
    if receipt.get("status") not in ("COMPLETE", "PARTIAL"):
        raise RuntimeError(f"ELENDRIL V8-X1 did not complete training/evaluation: {receipt}")
